<a href="https://colab.research.google.com/github/AlperYildirim1/crt-fourier-transformer-addition/blob/main/CRT_Oppenheim_Lim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# test9_cross_period_phase_chimera.py
# Colab standalone experiment for Pythia-6.9B
#
# Core question:
#   At the "=" token, can we combine:
#     - T2 sign from donor B
#     - T5 phase from donor C
#     - radii from magnitude donor A
#   and make the output units digit follow the CRT-composed target?
#
# Intervention at one layer only:
#   h_A' = h_A - Proj_{span(T2,T5)}(h_A)
#          + |c_A,T2| sign(c_B,T2) q2^T
#          + ||c_A,T5|| unit(c_C,T5) Q5^T
#
# Notes:
# - For integer sums, T2 is effectively one-dimensional:
#     sin(pi*s) = 0, cos(pi*s) = +/-1.
#   So T2 is treated as a sign coordinate, not a 2-D phase plane.
# - T5 is a genuine 2-D cos/sin plane.
# - The script uses single-layer interventions, not persistent steering.
# - First run with SMOKE_TEST=True.

# In Colab, run this once:
# !pip install -q transformers accelerate scikit-learn pandas tqdm

import gc
import json
import math
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


# =============================================================================
# Configuration
# =============================================================================

MODEL_NAME = "EleutherAI/pythia-6.9b"
OUT_DIR = "/content/pythia_test9_cross_period_chimera"

SEED = 42
RIDGE_ALPHA = 1.0
BATCH_SIZE = 12

# Smoke settings are intentionally small.
SMOKE_TEST = True
FIT_N = 600 if SMOKE_TEST else 4000
EVAL_N = 300 if SMOKE_TEST else 2000

# Fit planes over this range. Evaluation can use a subset.
FIT_LAYERS = list(range(10, 32))
EVAL_LAYERS = list(range(10, 32)) if not SMOKE_TEST else [10, 14, 18, 22, 26, 30, 31]

# Conditions:
# - noop: A radius + A T2 sign + A T5 phase (sanity; should preserve A)
# - consistent_B: A radii + B T2 sign + B T5 phase
# - cross_B2_C5: A radii + B T2 sign + C T5 phase (main CRT chimera)
# - cross_unit_radius: unit radii + B T2 sign + C T5 phase
CONDITIONS = [
    "noop",
    "consistent_B",
    "wrong_donor_B",
    "shuffled_phase_B",
    "cross_B2_C5",
    "cross_unit_radius",
]

# Restrict examples to single-token answers.
A_MAX = 99
B_MAX = 99


# =============================================================================
# Utilities
# =============================================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_last_nonpad_positions(attention_mask: torch.Tensor) -> torch.Tensor:
    """Works for either left or right padding."""
    positions = torch.arange(attention_mask.shape[1], device=attention_mask.device)
    return (attention_mask * positions.unsqueeze(0)).max(dim=1).values.long()


def pred_token_to_int(tokenizer, pred_id: int):
    text = tokenizer.decode([int(pred_id)]).strip()
    try:
        return int(text)
    except Exception:
        return None


def make_prompt(a: int, b: int) -> str:
    return f"Output ONLY a number. {a}+{b}="


def answer_token_id(tokenizer, n: int):
    ids = tokenizer(str(int(n)), add_special_tokens=False)["input_ids"]
    return int(ids[0]) if len(ids) == 1 else None


def get_blocks(model):
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    raise RuntimeError("Could not find transformer blocks.")


def orthonormalize_columns(M: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """QR with numerical rank truncation."""
    U, S, _ = np.linalg.svd(M, full_matrices=False)
    rank = int(np.sum(S > eps * max(float(S[0]), eps)))
    if rank == 0:
        raise RuntimeError("Matrix has zero numerical rank.")
    return U[:, :rank]


def crt_mod2_mod5_to_mod10(r2: int, r5: int) -> int:
    r2, r5 = int(r2) % 2, int(r5) % 5
    for u in range(10):
        if u % 2 == r2 and u % 5 == r5:
            return u
    raise RuntimeError("CRT composition failed.")


# =============================================================================
# Model and data
# =============================================================================

set_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()
model.config.pad_token_id = tokenizer.pad_token_id
blocks = get_blocks(model)
N_LAYERS = len(blocks)

FIT_LAYERS = [l for l in FIT_LAYERS if 0 <= l < N_LAYERS]
EVAL_LAYERS = [l for l in EVAL_LAYERS if l in FIT_LAYERS]
print("num layers:", N_LAYERS)
print("fit layers:", FIT_LAYERS)
print("eval layers:", EVAL_LAYERS)


def make_examples() -> List[dict]:
    rows = []
    for a in range(A_MAX + 1):
        for b in range(B_MAX + 1):
            s = a + b
            tid = answer_token_id(tokenizer, s)
            if tid is None:
                continue
            rows.append({
                "a": a,
                "b": b,
                "sum": s,
                "target_token_id": tid,
                "prompt": make_prompt(a, b),
            })
    return rows


@torch.no_grad()
def baseline_filter(rows: Sequence[dict]) -> List[dict]:
    kept = []
    for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="baseline filter"):
        batch = rows[start:start + BATCH_SIZE]
        enc = tokenizer(
            [x["prompt"] for x in batch],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)

        last = get_last_nonpad_positions(enc["attention_mask"])
        bi = torch.arange(len(batch), device=model.device)
        logits = model(**enc, use_cache=False).logits[bi, last]
        pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()

        for ex, pid in zip(batch, pred_ids):
            if int(pid) == int(ex["target_token_id"]):
                kept.append(ex)
    return kept


all_examples = make_examples()
print("candidate examples:", len(all_examples))
baseline_correct = baseline_filter(all_examples)
print("baseline correct:", len(baseline_correct), "/", len(all_examples))

rng = np.random.default_rng(SEED)
rng.shuffle(baseline_correct)

fit_rows = baseline_correct[:min(FIT_N, len(baseline_correct))]
pool_rows = baseline_correct[min(FIT_N, len(baseline_correct)):]
if len(pool_rows) < EVAL_N * 3:
    pool_rows = baseline_correct

print("fit rows:", len(fit_rows))
print("triplet source pool:", len(pool_rows))

# One-token numeric answer groups used for log-probability analysis.
# We aggregate probability over every available one-token integer with a
# particular units digit rather than relying only on argmax.
NUMERIC_TOKEN_BY_VALUE = {}
for _n in range(A_MAX + B_MAX + 1):
    _tid = answer_token_id(tokenizer, _n)
    if _tid is not None:
        NUMERIC_TOKEN_BY_VALUE[_n] = _tid

NUMERIC_VALUES = sorted(NUMERIC_TOKEN_BY_VALUE)
NUMERIC_TOKEN_IDS = torch.tensor(
    [NUMERIC_TOKEN_BY_VALUE[n] for n in NUMERIC_VALUES],
    dtype=torch.long,
    device=model.device,
)
NUMERIC_VALUE_TENSOR = torch.tensor(
    NUMERIC_VALUES,
    dtype=torch.long,
    device=model.device,
)
print("one-token numeric candidates:", len(NUMERIC_VALUES))


# =============================================================================
# Collect "=" residuals for fitting T2/T5 hypothesis-aligned directions
# =============================================================================

@torch.no_grad()
def collect_equal_residuals(rows: Sequence[dict], layers: Sequence[int]):
    store = {l: [] for l in layers}
    sums = []
    buffer: Dict[int, torch.Tensor] = {}

    def make_capture_hook(layer: int):
        def hook(module, inputs, output):
            hs = output[0] if isinstance(output, tuple) else output
            buffer[layer] = hs.detach()
        return hook

    handles = [blocks[l].register_forward_hook(make_capture_hook(l)) for l in layers]
    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc='collect "=" residuals'):
            batch = rows[start:start + BATCH_SIZE]
            enc = tokenizer(
                [x["prompt"] for x in batch],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)

            last = get_last_nonpad_positions(enc["attention_mask"])
            bi = torch.arange(len(batch), device=model.device)

            buffer.clear()
            model(**enc, use_cache=False)

            for l in layers:
                if l not in buffer:
                    raise RuntimeError(f"Missing layer {l} capture.")
                store[l].append(buffer[l][bi, last].float().cpu().numpy())

            sums.extend([int(x["sum"]) for x in batch])
    finally:
        for h in handles:
            h.remove()

    X = {l: np.concatenate(store[l], axis=0) for l in layers}
    return X, np.asarray(sums, dtype=np.float64)


X_equal, fit_sums = collect_equal_residuals(fit_rows, FIT_LAYERS)


@dataclass
class PeriodBases:
    q2: torch.Tensor   # [D, 1]
    q5: torch.Tensor   # [D, 2]
    q25: torch.Tensor  # [D, 3]


def fit_period_bases(X: np.ndarray, sums: np.ndarray) -> PeriodBases:
    """
    Fit hypothesis-aligned directions:
      T2: cos(pi*s) only (integer T2 is rank-1)
      T5: cos(2pi*s/5), sin(2pi*s/5)

    Then make T5 orthogonal to T2 so replacement components do not overlap.
    """
    y2 = np.cos(np.pi * sums).reshape(-1, 1)
    theta5 = 2.0 * np.pi * sums / 5.0
    y5 = np.column_stack([np.cos(theta5), np.sin(theta5)])

    reg2 = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True).fit(y2, X)
    reg5 = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True).fit(y5, X)

    w2 = reg2.coef_  # [D,1]
    w5 = reg5.coef_  # [D,2]

    q2 = orthonormalize_columns(w2)[:, :1]

    # Remove any component of the T5 plane inside T2, then orthonormalize.
    w5_orth = w5 - q2 @ (q2.T @ w5)
    q5 = orthonormalize_columns(w5_orth)
    if q5.shape[1] < 2:
        raise RuntimeError(f"T5 numerical rank < 2; got {q5.shape[1]}.")
    q5 = q5[:, :2]

    q25 = np.concatenate([q2, q5], axis=1)
    # Already mutually orthonormal up to numerical precision; QR cleans it.
    q25, _ = np.linalg.qr(q25)

    device = model.device
    return PeriodBases(
        q2=torch.tensor(q2, dtype=torch.float16, device=device),
        q5=torch.tensor(q5, dtype=torch.float16, device=device),
        q25=torch.tensor(q25[:, :3], dtype=torch.float16, device=device),
    )


bases_by_layer = {l: fit_period_bases(X_equal[l], fit_sums) for l in FIT_LAYERS}
del X_equal
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Fitted T2/T5 bases.")


# =============================================================================
# Build A/B/C triplets
# =============================================================================

def build_triplets(pool: Sequence[dict], n: int, seed: int) -> List[dict]:
    """
    Select:
      A = magnitude/radius donor and prompt actually run
      B = true phase donor for the consistent condition
      C = T5 phase donor for the cross-period condition
      D = deliberately wrong phase donor used as a matched null

    Main identifiability constraints:
      CRT(B mod2, C mod5) != A units
      D units != B units
    """
    rng = np.random.default_rng(seed)
    pool = list(pool)
    triplets = []
    attempts = 0
    max_attempts = max(10000, n * 400)

    while len(triplets) < n and attempts < max_attempts:
        attempts += 1
        ia, ib, ic, id_ = rng.choice(len(pool), size=4, replace=False)
        A, B, C, D = pool[ia], pool[ib], pool[ic], pool[id_]

        target_units = crt_mod2_mod5_to_mod10(B["sum"] % 2, C["sum"] % 5)
        if target_units == A["sum"] % 10:
            continue

        # Avoid a trivial cross condition.
        if B["sum"] % 2 == A["sum"] % 2 and C["sum"] % 5 == A["sum"] % 5:
            continue

        # Wrong donor must disagree with B's full units digit. Prefer that at
        # least one of its low-period residues also disagrees.
        if D["sum"] % 10 == B["sum"] % 10:
            continue
        if (D["sum"] % 2 == B["sum"] % 2) and (D["sum"] % 5 == B["sum"] % 5):
            continue

        triplets.append({
            "A": A,
            "B": B,
            "C": C,
            "D": D,
            "target_mod2": B["sum"] % 2,
            "target_mod5": C["sum"] % 5,
            "target_mod10": target_units,
            "consistent_B_mod10": B["sum"] % 10,
            "wrong_D_mod10": D["sum"] % 10,
        })

    if len(triplets) < n:
        raise RuntimeError(f"Could only build {len(triplets)} triplets after {attempts} attempts.")
    return triplets


triplets = build_triplets(pool_rows, min(EVAL_N, len(pool_rows) // 3), SEED + 9000)
print("triplets:", len(triplets))


# =============================================================================
# Layer-resolved donor capture and chimera intervention
# =============================================================================

@dataclass
class DonorCoords:
    # All tensors are [B,*], in the selected layer's orthonormal coordinate frames.
    a2: torch.Tensor
    b2: torch.Tensor
    d2: torch.Tensor
    a5: torch.Tensor
    b5: torch.Tensor
    c5: torch.Tensor
    d5: torch.Tensor


@torch.no_grad()
def capture_donor_coords(
    batch_triplets: Sequence[dict],
    layer: int,
    bases: PeriodBases,
) -> DonorCoords:
    """
    One clean pass over concatenated [A,B,C,D] prompts, capturing the "="
    residual after `layer`, then projecting into T2/T5 coordinates.
    """
    rows = (
        [t["A"] for t in batch_triplets]
        + [t["B"] for t in batch_triplets]
        + [t["C"] for t in batch_triplets]
        + [t["D"] for t in batch_triplets]
    )
    B = len(batch_triplets)
    captured = {}

    def hook(module, inputs, output):
        hs = output[0] if isinstance(output, tuple) else output
        captured["hs"] = hs.detach()

    handle = blocks[layer].register_forward_hook(hook)
    try:
        enc = tokenizer(
            [x["prompt"] for x in rows],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)
        last = get_last_nonpad_positions(enc["attention_mask"])
        bi = torch.arange(len(rows), device=model.device)
        model(**enc, use_cache=False)
        h = captured["hs"][bi, last].float()
    finally:
        handle.remove()

    hA = h[:B]
    hB = h[B:2 * B]
    hC = h[2 * B:3 * B]
    hD = h[3 * B:4 * B]

    q2 = bases.q2.float()
    q5 = bases.q5.float()

    return DonorCoords(
        a2=hA @ q2,
        b2=hB @ q2,
        d2=hD @ q2,
        a5=hA @ q5,
        b5=hB @ q5,
        c5=hC @ q5,
        d5=hD @ q5,
    )


def safe_unit_2d(z: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return z / (z.norm(dim=1, keepdim=True) + eps)


def build_target_contribution(
    coords: DonorCoords,
    bases: PeriodBases,
    condition: str,
) -> torch.Tensor:
    """
    Return target contribution in ambient residual space: [B,D].

    Nulls:
      wrong_donor_B:
        same A radii and same intervention machinery, but phase comes from D
        while evaluation still asks whether the output follows B.
      shuffled_phase_B:
        B phase coordinates are cyclically shifted across the batch, preserving
        their empirical distribution and intervention scale while breaking the
        example-to-donor correspondence.
    """
    q2 = bases.q2.float()
    q5 = bases.q5.float()

    if condition == "noop":
        new2 = coords.a2
        new5 = coords.a5

    elif condition == "consistent_B":
        new2 = coords.a2.abs() * torch.sign(coords.b2)
        new5 = coords.a5.norm(dim=1, keepdim=True) * safe_unit_2d(coords.b5)

    elif condition == "wrong_donor_B":
        new2 = coords.a2.abs() * torch.sign(coords.d2)
        new5 = coords.a5.norm(dim=1, keepdim=True) * safe_unit_2d(coords.d5)

    elif condition == "shuffled_phase_B":
        # Deterministic cyclic shift. For B=1, fall back to the explicit D null.
        if coords.b2.shape[0] > 1:
            shuf_b2 = torch.roll(coords.b2, shifts=1, dims=0)
            shuf_b5 = torch.roll(coords.b5, shifts=1, dims=0)
        else:
            shuf_b2, shuf_b5 = coords.d2, coords.d5
        new2 = coords.a2.abs() * torch.sign(shuf_b2)
        new5 = coords.a5.norm(dim=1, keepdim=True) * safe_unit_2d(shuf_b5)

    elif condition == "cross_B2_C5":
        new2 = coords.a2.abs() * torch.sign(coords.b2)
        new5 = coords.a5.norm(dim=1, keepdim=True) * safe_unit_2d(coords.c5)

    elif condition == "cross_unit_radius":
        new2 = torch.sign(coords.b2)
        new5 = safe_unit_2d(coords.c5)

    else:
        raise ValueError(f"Unknown condition: {condition}")

    return new2 @ q2.T + new5 @ q5.T


# Hook globals for one evaluation pass.
_INTERVENTION_TARGET = None
_INTERVENTION_POS = None
_INTERVENTION_BI = None
_INTERVENTION_Q25 = None


def make_single_layer_chimera_hook():
    def hook(module, inputs, output):
        global _INTERVENTION_TARGET, _INTERVENTION_POS, _INTERVENTION_BI, _INTERVENTION_Q25

        if isinstance(output, tuple):
            hs = output[0].clone()
            rest = tuple(output[1:])
        else:
            hs = output.clone()
            rest = None

        v = hs[_INTERVENTION_BI, _INTERVENTION_POS]
        Q = _INTERVENTION_Q25.to(dtype=v.dtype)
        target = _INTERVENTION_TARGET.to(dtype=v.dtype)

        # Remove A's existing T2/T5 content and replace with the chimera.
        hs[_INTERVENTION_BI, _INTERVENTION_POS] = v - (v @ Q) @ Q.T + target

        return hs if rest is None else (hs,) + rest

    return hook


def analysis_target_mod10(t: dict, condition: str) -> int:
    """
    Target used for phase-importance analysis.

    For noop/consistent/null conditions, evaluate the same B units-digit target,
    making their log-probabilities directly paired and comparable.
    For cross conditions, use the CRT-composed B-mod2/C-mod5 target.
    """
    if condition in {"noop", "consistent_B", "wrong_donor_B", "shuffled_phase_B"}:
        return int(t["consistent_B_mod10"])
    return int(t["target_mod10"])


def numeric_units_logprob(logits_row: torch.Tensor, units_digit: int) -> float:
    """
    Log probability mass assigned to all available one-token numeric answers
    with the requested units digit.
    """
    numeric_logits = logits_row[NUMERIC_TOKEN_IDS]
    mask = (NUMERIC_VALUE_TENSOR % 10) == int(units_digit)
    if not bool(mask.any()):
        return float("nan")
    log_num = torch.logsumexp(numeric_logits[mask].float(), dim=0)
    log_den = torch.logsumexp(logits_row.float(), dim=0)
    return float((log_num - log_den).item())


@torch.no_grad()
def evaluate_layer_condition(
    triplets: Sequence[dict],
    layer: int,
    condition: str,
) -> Tuple[dict, List[dict]]:
    global _INTERVENTION_TARGET, _INTERVENTION_POS, _INTERVENTION_BI, _INTERVENTION_Q25

    bases = bases_by_layer[layer]
    counts = {
        "n": 0,
        "parseable": 0,
        "follow_target_mod2": 0,
        "follow_target_mod5": 0,
        "follow_target_mod10": 0,
        "stay_A_mod10": 0,
        "follow_B_mod10": 0,
        "exact_A": 0,
        "sum_target_logprob": 0.0,
        "sum_A_logprob": 0.0,
    }
    records = []

    for start in tqdm(
        range(0, len(triplets), BATCH_SIZE),
        desc=f"L{layer} {condition}",
        leave=False,
    ):
        batch = triplets[start:start + BATCH_SIZE]
        A_rows = [t["A"] for t in batch]

        coords = capture_donor_coords(batch, layer, bases)
        target = build_target_contribution(coords, bases, condition)

        enc = tokenizer(
            [x["prompt"] for x in A_rows],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)
        last = get_last_nonpad_positions(enc["attention_mask"])
        bi = torch.arange(len(batch), device=model.device)

        _INTERVENTION_TARGET = target
        _INTERVENTION_POS = last
        _INTERVENTION_BI = bi
        _INTERVENTION_Q25 = bases.q25

        handle = blocks[layer].register_forward_hook(make_single_layer_chimera_hook())
        try:
            logits = model(**enc, use_cache=False).logits[bi, last].float()
            pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()
        finally:
            handle.remove()

        for t, pid, logits_row in zip(batch, pred_ids, logits):
            A, B, C, D = t["A"], t["B"], t["C"], t["D"]
            analysis_target = analysis_target_mod10(t, condition)
            target_logprob = numeric_units_logprob(logits_row, analysis_target)
            A_logprob = numeric_units_logprob(logits_row, A["sum"] % 10)
            counts["sum_target_logprob"] += target_logprob
            counts["sum_A_logprob"] += A_logprob
            pred = pred_token_to_int(tokenizer, pid)

            counts["n"] += 1
            counts["exact_A"] += int(int(pid) == int(A["target_token_id"]))

            rec = {
                "layer": layer,
                "condition": condition,
                "a_A": A["a"],
                "b_A": A["b"],
                "sum_A": A["sum"],
                "sum_B": B["sum"],
                "sum_C": C["sum"],
                "sum_D": D["sum"],
                "analysis_target_mod10": analysis_target,
                "target_units_logprob": target_logprob,
                "A_units_logprob": A_logprob,
                "target_vs_A_logprob": target_logprob - A_logprob,
                "target_mod2": t["target_mod2"],
                "target_mod5": t["target_mod5"],
                "target_mod10": t["target_mod10"],
                "pred": pred,
            }

            if pred is None:
                rec["parseable"] = 0
                records.append(rec)
                continue

            counts["parseable"] += 1
            counts["follow_target_mod2"] += int(pred % 2 == t["target_mod2"])
            counts["follow_target_mod5"] += int(pred % 5 == t["target_mod5"])
            counts["follow_target_mod10"] += int(pred % 10 == t["target_mod10"])
            counts["stay_A_mod10"] += int(pred % 10 == A["sum"] % 10)
            counts["follow_B_mod10"] += int(pred % 10 == B["sum"] % 10)

            rec.update({
                "parseable": 1,
                "pred_mod2": pred % 2,
                "pred_mod5": pred % 5,
                "pred_mod10": pred % 10,
                "follow_target_mod2": int(pred % 2 == t["target_mod2"]),
                "follow_target_mod5": int(pred % 5 == t["target_mod5"]),
                "follow_target_mod10": int(pred % 10 == t["target_mod10"]),
                "stay_A_mod10": int(pred % 10 == A["sum"] % 10),
                "follow_B_mod10": int(pred % 10 == B["sum"] % 10),
            })
            records.append(rec)

    n = max(counts["n"], 1)
    summary = {
        "layer": layer,
        "condition": condition,
        "n": counts["n"],
        "parseable": counts["parseable"] / n,
        "exact_A": counts["exact_A"] / n,
        "follow_target_mod2": counts["follow_target_mod2"] / n,
        "follow_target_mod5": counts["follow_target_mod5"] / n,
        "follow_target_mod10": counts["follow_target_mod10"] / n,
        "stay_A_mod10": counts["stay_A_mod10"] / n,
        "follow_B_mod10": counts["follow_B_mod10"] / n,
        "net_target_vs_stay10": (
            counts["follow_target_mod10"] - counts["stay_A_mod10"]
        ) / n,
        "mean_target_units_logprob": counts["sum_target_logprob"] / n,
        "mean_A_units_logprob": counts["sum_A_logprob"] / n,
        "mean_target_vs_A_logprob": (
            counts["sum_target_logprob"] - counts["sum_A_logprob"]
        ) / n,
    }
    return summary, records


# =============================================================================
# Run
# =============================================================================

all_summaries = []
all_records = []

for layer in EVAL_LAYERS:
    print("\n" + "=" * 100)
    print("LAYER", layer)
    print("=" * 100)

    for condition in CONDITIONS:
        summary, records = evaluate_layer_condition(triplets, layer, condition)
        all_summaries.append(summary)
        all_records.extend(records)
        print(summary)

        pd.DataFrame(all_summaries).to_csv(
            os.path.join(OUT_DIR, "test9_summary_partial.csv"),
            index=False,
        )
        pd.DataFrame(all_records).to_csv(
            os.path.join(OUT_DIR, "test9_predictions_partial.csv"),
            index=False,
        )

summary_df = pd.DataFrame(all_summaries)
pred_df = pd.DataFrame(all_records)

# Paired phase-importance contrasts. Because every condition uses the same
# ordered triplets, row-wise subtraction estimates how much the correct B phase
# raises B-target log probability above matched null interventions.
paired_rows = []
phase_conditions = ["noop", "wrong_donor_B", "shuffled_phase_B"]
for layer in EVAL_LAYERS:
    layer_df = pred_df[pred_df["layer"] == layer].copy()
    true_df = (
        layer_df[layer_df["condition"] == "consistent_B"]
        .reset_index(drop=True)
    )
    if true_df.empty:
        continue

    for null_condition in phase_conditions:
        null_df = (
            layer_df[layer_df["condition"] == null_condition]
            .reset_index(drop=True)
        )
        if len(null_df) != len(true_df):
            continue

        delta_lp = (
            true_df["target_units_logprob"].to_numpy()
            - null_df["target_units_logprob"].to_numpy()
        )
        true_follow = (
            (true_df["pred"].notna())
            & ((true_df["pred"].astype("Int64") % 10)
               == true_df["analysis_target_mod10"].astype("Int64"))
        ).astype(float).to_numpy()
        null_follow = (
            (null_df["pred"].notna())
            & ((null_df["pred"].astype("Int64") % 10)
               == null_df["analysis_target_mod10"].astype("Int64"))
        ).astype(float).to_numpy()

        # Nonparametric bootstrap CI for mean paired log-probability lift.
        boot_rng = np.random.default_rng(SEED + 100000 + layer)
        boot_means = []
        if len(delta_lp):
            for _ in range(2000):
                idx = boot_rng.integers(0, len(delta_lp), size=len(delta_lp))
                boot_means.append(float(np.mean(delta_lp[idx])))
            ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
        else:
            ci_low = ci_high = float("nan")

        paired_rows.append({
            "layer": layer,
            "true_condition": "consistent_B",
            "null_condition": null_condition,
            "n": len(delta_lp),
            "mean_delta_target_logprob": float(np.mean(delta_lp)),
            "median_delta_target_logprob": float(np.median(delta_lp)),
            "bootstrap_ci95_low": float(ci_low),
            "bootstrap_ci95_high": float(ci_high),
            "follow_B_true": float(np.mean(true_follow)),
            "follow_B_null": float(np.mean(null_follow)),
            "follow_B_lift": float(np.mean(true_follow - null_follow)),
        })

phase_null_df = pd.DataFrame(paired_rows)

summary_path = os.path.join(OUT_DIR, "test9_cross_period_phase_chimera_summary.csv")
pred_path = os.path.join(OUT_DIR, "test9_cross_period_phase_chimera_predictions.csv")
null_path = os.path.join(OUT_DIR, "test9_phase_null_contrasts.csv")
json_path = os.path.join(OUT_DIR, "test9_config.json")

summary_df.to_csv(summary_path, index=False)
pred_df.to_csv(pred_path, index=False)
phase_null_df.to_csv(null_path, index=False)
with open(json_path, "w") as f:
    json.dump({
        "model": MODEL_NAME,
        "seed": SEED,
        "ridge_alpha": RIDGE_ALPHA,
        "fit_n": len(fit_rows),
        "eval_n": len(triplets),
        "fit_layers": FIT_LAYERS,
        "eval_layers": EVAL_LAYERS,
        "conditions": CONDITIONS,
        "smoke_test": SMOKE_TEST,
    }, f, indent=2)

print("\n" + "#" * 100)
print("FINAL SUMMARY")
print("#" * 100)
print(summary_df.to_string(index=False))
print("\nPHASE NULL CONTRASTS")
print(phase_null_df.to_string(index=False))
print("\nsaved:", summary_path)
print("saved:", pred_path)
print("saved:", null_path)
print("saved:", json_path)


# =============================================================================
# Persistent multi-layer phase transplantation
# =============================================================================
#
# These experiments repeat the same phase/sign replacement at every layer in a
# selected range. Donor coordinates are captured from CLEAN A/B/C/D forward
# passes at each layer, while the actual A prompt is intervened on sequentially
# through all selected layers.
#
# This directly tests whether the gap between single-shot phase transplantation
# and earlier persistent steering is caused by downstream recomputation.
#
# Default smoke ranges:
#   26 -> final fitted layer
#   18 -> final fitted layer
#
# Conditions:
#   consistent_B  : A radii + B T2/T5 phase at every intervention layer
#   cross_B2_C5   : A radii + B T2 sign + C T5 phase at every layer

PERSISTENT_START_LAYERS = [26, 18]
PERSISTENT_CONDITIONS = ["consistent_B", "cross_B2_C5"]


@torch.no_grad()
def capture_coords_for_layers(
    batch_triplets: Sequence[dict],
    layers: Sequence[int],
) -> Dict[int, DonorCoords]:
    """
    Capture clean donor coordinates for all requested layers in one forward pass
    over concatenated [A,B,C,D] prompts.
    """
    rows = (
        [t["A"] for t in batch_triplets]
        + [t["B"] for t in batch_triplets]
        + [t["C"] for t in batch_triplets]
        + [t["D"] for t in batch_triplets]
    )
    B = len(batch_triplets)
    captured: Dict[int, torch.Tensor] = {}

    def make_hook(layer: int):
        def hook(module, inputs, output):
            hs = output[0] if isinstance(output, tuple) else output
            captured[layer] = hs.detach()
        return hook

    handles = [blocks[l].register_forward_hook(make_hook(l)) for l in layers]
    try:
        enc = tokenizer(
            [x["prompt"] for x in rows],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)
        last = get_last_nonpad_positions(enc["attention_mask"])
        bi = torch.arange(len(rows), device=model.device)
        model(**enc, use_cache=False)
    finally:
        for h in handles:
            h.remove()

    out: Dict[int, DonorCoords] = {}
    for layer in layers:
        h = captured[layer]
        hA = h[:B][bi[:B], last[:B]].float()
        hB = h[B:2 * B][bi[:B], last[B:2 * B]].float()
        hC = h[2 * B:3 * B][bi[:B], last[2 * B:3 * B]].float()
        hD = h[3 * B:4 * B][bi[:B], last[3 * B:4 * B]].float()

        bases = bases_by_layer[layer]
        q2 = bases.q2.float()
        q5 = bases.q5.float()

        out[layer] = DonorCoords(
            a2=hA @ q2,
            b2=hB @ q2,
            d2=hD @ q2,
            a5=hA @ q5,
            b5=hB @ q5,
            c5=hC @ q5,
            d5=hD @ q5,
        )
    return out


def make_persistent_layer_hook(
    target: torch.Tensor,
    positions: torch.Tensor,
    batch_indices: torch.Tensor,
    q25: torch.Tensor,
):
    """
    Create a layer-local replacement hook. Each layer has its own learned basis
    and donor-derived target contribution.
    """
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            hs = output[0].clone()
            rest = tuple(output[1:])
        else:
            hs = output.clone()
            rest = None

        v = hs[batch_indices, positions]
        Q = q25.to(dtype=v.dtype)
        tgt = target.to(dtype=v.dtype)

        hs[batch_indices, positions] = v - (v @ Q) @ Q.T + tgt
        return hs if rest is None else (hs,) + rest

    return hook


@torch.no_grad()
def evaluate_persistent_condition(
    triplets: Sequence[dict],
    start_layer: int,
    condition: str,
) -> Tuple[dict, List[dict]]:
    """
    Re-apply the phase/sign transplant at every fitted layer from start_layer to
    the final fitted layer.
    """
    intervention_layers = [l for l in FIT_LAYERS if l >= start_layer]
    if not intervention_layers:
        raise ValueError(f"No fitted layers at or after {start_layer}.")

    counts = {
        "n": 0,
        "parseable": 0,
        "follow_target_mod2": 0,
        "follow_target_mod5": 0,
        "follow_target_mod10": 0,
        "stay_A_mod10": 0,
        "follow_B_mod10": 0,
        "exact_A": 0,
        "sum_target_logprob": 0.0,
        "sum_A_logprob": 0.0,
    }
    records = []

    for start in tqdm(
        range(0, len(triplets), BATCH_SIZE),
        desc=f"PERSIST {start_layer}->{intervention_layers[-1]} {condition}",
        leave=False,
    ):
        batch = triplets[start:start + BATCH_SIZE]
        A_rows = [t["A"] for t in batch]

        coords_by_layer = capture_coords_for_layers(batch, intervention_layers)

        enc = tokenizer(
            [x["prompt"] for x in A_rows],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)
        last = get_last_nonpad_positions(enc["attention_mask"])
        bi = torch.arange(len(batch), device=model.device)

        handles = []
        try:
            for layer in intervention_layers:
                bases = bases_by_layer[layer]
                target = build_target_contribution(
                    coords_by_layer[layer],
                    bases,
                    condition,
                )
                handles.append(
                    blocks[layer].register_forward_hook(
                        make_persistent_layer_hook(
                            target=target,
                            positions=last,
                            batch_indices=bi,
                            q25=bases.q25,
                        )
                    )
                )

            logits = model(**enc, use_cache=False).logits[bi, last].float()
            pred_ids = logits.argmax(dim=-1).detach().cpu().tolist()
        finally:
            for h in handles:
                h.remove()

        for t, pid, logits_row in zip(batch, pred_ids, logits):
            A, B, C, D = t["A"], t["B"], t["C"], t["D"]
            pred = pred_token_to_int(tokenizer, pid)
            analysis_target = analysis_target_mod10(t, condition)

            target_logprob = numeric_units_logprob(logits_row, analysis_target)
            A_logprob = numeric_units_logprob(logits_row, A["sum"] % 10)

            counts["n"] += 1
            counts["exact_A"] += int(int(pid) == int(A["target_token_id"]))
            counts["sum_target_logprob"] += target_logprob
            counts["sum_A_logprob"] += A_logprob

            rec = {
                "mode": "persistent",
                "start_layer": start_layer,
                "end_layer": intervention_layers[-1],
                "n_intervention_layers": len(intervention_layers),
                "condition": condition,
                "a_A": A["a"],
                "b_A": A["b"],
                "sum_A": A["sum"],
                "sum_B": B["sum"],
                "sum_C": C["sum"],
                "sum_D": D["sum"],
                "analysis_target_mod10": analysis_target,
                "target_units_logprob": target_logprob,
                "A_units_logprob": A_logprob,
                "target_vs_A_logprob": target_logprob - A_logprob,
                "target_mod2": t["target_mod2"],
                "target_mod5": t["target_mod5"],
                "target_mod10": t["target_mod10"],
                "pred": pred,
            }

            if pred is None:
                rec["parseable"] = 0
                records.append(rec)
                continue

            counts["parseable"] += 1
            counts["follow_target_mod2"] += int(pred % 2 == t["target_mod2"])
            counts["follow_target_mod5"] += int(pred % 5 == t["target_mod5"])
            counts["follow_target_mod10"] += int(pred % 10 == t["target_mod10"])
            counts["stay_A_mod10"] += int(pred % 10 == A["sum"] % 10)
            counts["follow_B_mod10"] += int(pred % 10 == B["sum"] % 10)

            rec.update({
                "parseable": 1,
                "pred_mod2": pred % 2,
                "pred_mod5": pred % 5,
                "pred_mod10": pred % 10,
                "follow_target_mod2": int(pred % 2 == t["target_mod2"]),
                "follow_target_mod5": int(pred % 5 == t["target_mod5"]),
                "follow_target_mod10": int(pred % 10 == t["target_mod10"]),
                "stay_A_mod10": int(pred % 10 == A["sum"] % 10),
                "follow_B_mod10": int(pred % 10 == B["sum"] % 10),
            })
            records.append(rec)

    n = max(counts["n"], 1)
    summary = {
        "mode": "persistent",
        "start_layer": start_layer,
        "end_layer": intervention_layers[-1],
        "n_intervention_layers": len(intervention_layers),
        "condition": condition,
        "n": counts["n"],
        "parseable": counts["parseable"] / n,
        "exact_A": counts["exact_A"] / n,
        "follow_target_mod2": counts["follow_target_mod2"] / n,
        "follow_target_mod5": counts["follow_target_mod5"] / n,
        "follow_target_mod10": counts["follow_target_mod10"] / n,
        "stay_A_mod10": counts["stay_A_mod10"] / n,
        "follow_B_mod10": counts["follow_B_mod10"] / n,
        "net_target_vs_stay10": (
            counts["follow_target_mod10"] - counts["stay_A_mod10"]
        ) / n,
        "mean_target_units_logprob": counts["sum_target_logprob"] / n,
        "mean_A_units_logprob": counts["sum_A_logprob"] / n,
        "mean_target_vs_A_logprob": (
            counts["sum_target_logprob"] - counts["sum_A_logprob"]
        ) / n,
    }
    return summary, records


persistent_summaries = []
persistent_records = []

for start_layer in PERSISTENT_START_LAYERS:
    for condition in PERSISTENT_CONDITIONS:
        summary, records = evaluate_persistent_condition(
            triplets=triplets,
            start_layer=start_layer,
            condition=condition,
        )
        persistent_summaries.append(summary)
        persistent_records.extend(records)
        print("\nPERSISTENT RESULT")
        print(summary)

        pd.DataFrame(persistent_summaries).to_csv(
            os.path.join(OUT_DIR, "test9_persistent_summary_partial.csv"),
            index=False,
        )
        pd.DataFrame(persistent_records).to_csv(
            os.path.join(OUT_DIR, "test9_persistent_predictions_partial.csv"),
            index=False,
        )

persistent_summary_df = pd.DataFrame(persistent_summaries)
persistent_pred_df = pd.DataFrame(persistent_records)

persistent_summary_path = os.path.join(
    OUT_DIR,
    "test9_persistent_phase_transplant_summary.csv",
)
persistent_pred_path = os.path.join(
    OUT_DIR,
    "test9_persistent_phase_transplant_predictions.csv",
)

persistent_summary_df.to_csv(persistent_summary_path, index=False)
persistent_pred_df.to_csv(persistent_pred_path, index=False)

# Direct comparison against the corresponding single-shot result at the same
# start layer. For consistent_B use follow_B_mod10; for cross use CRT mod10.
comparison_rows = []
for row in persistent_summaries:
    start_layer = int(row["start_layer"])
    condition = row["condition"]

    single = summary_df[
        (summary_df["layer"] == start_layer)
        & (summary_df["condition"] == condition)
    ]
    if single.empty:
        continue
    single = single.iloc[0]

    metric = "follow_B_mod10" if condition == "consistent_B" else "follow_target_mod10"
    comparison_rows.append({
        "start_layer": start_layer,
        "end_layer": row["end_layer"],
        "condition": condition,
        "metric": metric,
        "single_shot": float(single[metric]),
        "persistent": float(row[metric]),
        "persistent_minus_single": float(row[metric] - single[metric]),
        "single_stay_A_mod10": float(single["stay_A_mod10"]),
        "persistent_stay_A_mod10": float(row["stay_A_mod10"]),
        "stay_A_change": float(row["stay_A_mod10"] - single["stay_A_mod10"]),
        "single_mean_target_logprob": float(single["mean_target_units_logprob"]),
        "persistent_mean_target_logprob": float(row["mean_target_units_logprob"]),
        "target_logprob_change": float(
            row["mean_target_units_logprob"] - single["mean_target_units_logprob"]
        ),
    })

persistent_comparison_df = pd.DataFrame(comparison_rows)
persistent_comparison_path = os.path.join(
    OUT_DIR,
    "test9_persistent_vs_single_comparison.csv",
)
persistent_comparison_df.to_csv(persistent_comparison_path, index=False)

print("\n" + "#" * 100)
print("PERSISTENT PHASE TRANSPLANT SUMMARY")
print("#" * 100)
print(persistent_summary_df.to_string(index=False))

print("\nPERSISTENT VS SINGLE-SHOT")
print(persistent_comparison_df.to_string(index=False))

print("\nsaved:", persistent_summary_path)
print("saved:", persistent_pred_path)
print("saved:", persistent_comparison_path)